# Structured Output with Amazon Bedrock

Two approaches for structured output:
1. **ChatBedrockConverse** (recommended): Native structured output for ALL models
2. **ChatBedrock**: Requires JsonOutputParser workaround

## Prerequisites
```bash
# Install dependencies
uv sync

# Configure AWS credentials
export AWS_PROFILE=your-profile
```

## 1. Setup and Imports

In [1]:
from typing import List

from pydantic import BaseModel, Field

from config import Model
from structured_output import get_llm

# Show available models
print("Available models:")
for model in Model:
    print(f"  - {model.name}: {model.value.name}")

Available models:
  - NOVA_MICRO: Amazon Nova Micro
  - NOVA_LITE: Amazon Nova Lite
  - NOVA_PRO: Amazon Nova Pro
  - NOVA_PREMIER: Amazon Nova Premier
  - CLAUDE_HAIKU: Claude 4.5 Haiku
  - CLAUDE_SONNET: Claude 4.5 Sonnet
  - GPT_OSS_20B: GPT-OSS 20B
  - GPT_OSS_120B: GPT-OSS 120B


## 2. ChatBedrockConverse - Native Structured Output (Recommended)

In [2]:
# Test with Nova
print("=" * 60)
print("NOVA with ChatBedrockConverse (native support)")
print("=" * 60)

llm = get_llm(Model.NOVA_MICRO)  # Default: use_bedrock_converse=True
result = llm.invoke("Should startups use microservices or monoliths?")

print(f"Type: {type(result)}")
print(f"Answer: {result.answer}")
print(f"Confidence: {result.confidence:.2%}")

NOVA with ChatBedrockConverse (native support)
Type: <class 'structured_output.Response'>
Answer: It depends on the specific needs and goals of the startup.
Confidence: 90.00%


In [3]:
# Test with Claude
print("=" * 60)
print("CLAUDE with ChatBedrockConverse (native support)")
print("=" * 60)

llm = get_llm(Model.CLAUDE_HAIKU)
result = llm.invoke("Is Python good for machine learning?")

print(f"Type: {type(result)}")
print(f"Answer: {result.answer}")
print(f"Confidence: {result.confidence:.2%}")

CLAUDE with ChatBedrockConverse (native support)
Type: <class 'structured_output.Response'>
Answer: Yes, Python is excellent for machine learning and is the most popular language for ML applications. It offers powerful libraries, ease of use, strong community support, and is the de facto standard in both research and production environments.
Confidence: 95.00%


## 3. ChatBedrock - JsonOutputParser Workaround

For legacy compatibility or when ChatBedrockConverse is not available.

In [4]:
print("=" * 60)
print("NOVA LITE with ChatBedrock (JsonOutputParser)")
print("=" * 60)

llm = get_llm(Model.NOVA_LITE, use_bedrock_converse=False)

# Note: Different invoke format - needs dict with 'query' key
result = llm.invoke({"query": "Should startups use microservices or monoliths?"})

print(f"Type: {type(result)}")
print(f"Answer: {result['answer']}")  # Dict access
print(f"Confidence: {result['confidence']:.2%}")

NOVA LITE with ChatBedrock (JsonOutputParser)
Type: <class 'dict'>
Answer: It depends on the specific needs and goals of the startup. Microservices are recommended for large, complex applications that require scalability and flexibility, while monoliths are better suited for smaller applications or startups with limited resources.
Confidence: 90.00%


## 4. Custom Output Models

Use any Pydantic model for structured output.

In [5]:
# Define a custom model
class TechnicalAnalysis(BaseModel):
    topic: str = Field(description="The technology being analyzed")
    pros: List[str] = Field(description="Advantages")
    cons: List[str] = Field(description="Disadvantages")
    recommendation: str = Field(description="Final recommendation")
    use_cases: List[str] = Field(description="Best use cases")


# Use custom model with ChatBedrockConverse
llm = get_llm(Model.NOVA_LITE, output=TechnicalAnalysis)
result = llm.invoke("Analyze Kubernetes for production deployments")

print(f"Topic: {result.topic}")
print("\nPros:")
for pro in result.pros:
    print(f"  + {pro}")
print("\nCons:")
for con in result.cons:
    print(f"  - {con}")
print(f"\nRecommendation: {result.recommendation}")

Topic: Kubernetes

Pros:
  + Highly scalable
  + Automated deployment
  + Robust ecosystem
  + Flexible

Cons:
  - Complexity
  - Learning curve
  - Resource intensive

Recommendation: Recommended


In [6]:
# Same custom model with ChatBedrock (legacy)
llm = get_llm(Model.NOVA_MICRO, output=TechnicalAnalysis, use_bedrock_converse=False)
result = llm.invoke({"query": "Analyze Docker for development environments"})

print(f"Topic: {result['topic']}")  # Dict access for ChatBedrock
print(f"Pros: {result['pros'][:2]}...")  # First 2 pros
print(f"Recommendation: {result['recommendation']}")

Topic: Docker for Development Environments
Pros: ['Consistency across different environments', 'Isolation of dependencies']...
Recommendation: Use Docker for development environments where consistency, isolation, and portability are key


## 5. Side-by-Side Comparison

In [7]:
test_prompt = "Is TypeScript better than JavaScript?"

# Test both approaches with the same model
model = Model.NOVA_LITE

print("=" * 60)
print("ChatBedrockConverse")
print("=" * 60)
llm_converse = get_llm(model, use_bedrock_converse=True)
result_converse = llm_converse.invoke(test_prompt)
print(f"Returns Pydantic model: {type(result_converse).__name__}")
print(f"Answer: {result_converse.answer[:50]}...")

print("\n" + "=" * 60)
print("ChatBedrock")
print("=" * 60)
llm_bedrock = get_llm(model, use_bedrock_converse=False)
result_bedrock = llm_bedrock.invoke({"query": test_prompt})
print(f"Returns dict: {type(result_bedrock).__name__}")
print(f"Answer: {result_bedrock['answer'][:50]}...")

ChatBedrockConverse
Returns Pydantic model: Response
Answer: TypeScript and JavaScript serve different purposes...

ChatBedrock
Returns dict: dict
Answer: It depends on the specific needs of the project....


## 6. Export Test Script

In [8]:
%%writefile test_structured_output.py
#!/usr/bin/env python3
"""
Test script for Bedrock structured output approaches.
"""

from config import Model
from structured_output import get_llm


def test_approaches():
    prompt = "Should we use containers in production?"

    print("\n" + "=" * 60)
    print("Testing ChatBedrockConverse (native support)")
    print("=" * 60)

    # Native structured output
    llm = get_llm(Model.NOVA_MICRO, use_bedrock_converse=True)
    result = llm.invoke(prompt)
    print(f"Type: {type(result).__name__} (Pydantic model)")
    print(f"Answer: {result.answer[:50]}...")

    print("\n" + "=" * 60)
    print("Testing ChatBedrock (JsonOutputParser)")
    print("=" * 60)

    # JsonOutputParser workaround
    llm = get_llm(Model.NOVA_MICRO, use_bedrock_converse=False)
    result = llm.invoke({"query": prompt})
    print(f"Type: {type(result).__name__} (dictionary)")
    print(f"Answer: {result['answer'][:50]}...")


if __name__ == "__main__":
    test_approaches()
    print("\n[SUCCESS] Both approaches work")

Overwriting test_structured_output.py


## Key Takeaways

### Simple Rule
- **ChatBedrockConverse**: Native `with_structured_output()`
- **ChatBedrock**: Requires JsonOutputParser

### Usage Pattern
```python
from structured_output import get_llm
from config import Model

# Recommended: ChatBedrockConverse (default)
llm = get_llm(Model.NOVA_MICRO)
result = llm.invoke("prompt")  # Returns Pydantic model

# Legacy: ChatBedrock with JsonOutputParser
llm = get_llm(Model.NOVA_MICRO, use_bedrock_converse=False)
result = llm.invoke({"query": "prompt"})  # Returns dict

# Custom output model
llm = get_llm(Model.CLAUDE_HAIKU, output=MyModel)
```